# 01 — Download SportsMOT and convert to YOLO format

**Run this on a free T4 runtime.** It does no model training, only data
download and reformatting, so there is no reason to spend A100 hours on it.

This notebook will:
1. Install the project + its dependencies.
2. Mount your Google Drive (for caching the dataset across sessions).
3. Download SportsMOT.
4. Filter to the basketball subset and convert it to YOLO format using
   `src.data.mot_to_yolo`.
5. Sanity-check the result: print stats, list a few label files, render
   one annotated frame.

After this notebook completes, switch the runtime to **A100** and run
`03_train_yolo.ipynb` to fine-tune YOLO11s on the converted dataset.

## 1. Environment setup

Detect whether we are on Colab, install dependencies, and clone the
project repo if needed.

In [ ]:
import os, sys, subprocess

ON_COLAB = 'google.colab' in sys.modules
print(f'On Colab: {ON_COLAB}')

if ON_COLAB:
    # Clone the project repo into /content if it isn't there yet.
    REPO_URL  = os.environ.get('STAT_TRACKING_REPO', 'https://github.com/shreyas0328/stat_tracking.git')
    REPO_DIR  = '/content/stat_tracking'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    # Local run: assume you already `pip install -r requirements.txt`.
    pass

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
# Mount Google Drive so we can cache the dataset across runtime restarts.
# We will write the converted YOLO-format dataset to Drive once and reuse it
# in every subsequent training notebook.

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/stat_tracking'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
else:
    DRIVE_ROOT = os.path.abspath('data')
    os.makedirs(DRIVE_ROOT, exist_ok=True)

SPORTSMOT_RAW = os.path.join(DRIVE_ROOT, 'sportsmot_raw')
SPORTSMOT_YOLO = os.path.join(DRIVE_ROOT, 'sportsmot_yolo')
print('raw  :', SPORTSMOT_RAW)
print('yolo :', SPORTSMOT_YOLO)

## 2. Download SportsMOT (basketball-relevant TARs only)

The Hugging Face mirror packages each split as a single TAR file:

| File | Size | Contains |
|------|-----:|----------|
| `dataset/train.tar` | 6.7 GB | training sequences (all sports interleaved) |
| `dataset/val.tar`   | 6.6 GB | val sequences (used for our metrics) |
| `dataset/test.tar`  | 23 GB  | held-out test set, **no annotations** — we skip |

So we download `train.tar + val.tar + splits_txt/` (~13 GB), extract them
in place, and then run the YOLO-format conversion which filters down to
basketball-only. First run takes ~10–15 minutes on Colab depending on
bandwidth; subsequent runs are instant because everything caches to your
mounted Drive.

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

raw = Path(SPORTSMOT_RAW)
raw.mkdir(parents=True, exist_ok=True)

# Already extracted on a previous run? Skip both download and extract.
already_extracted = (raw / 'dataset' / 'train').is_dir() and (raw / 'dataset' / 'val').is_dir()
print('Already extracted?', already_extracted)

if not already_extracted:
    print('Downloading splits_txt + train.tar + val.tar (~13 GB)...')
    snapshot_download(
        repo_id='MCG-NJU/SportsMOT',
        repo_type='dataset',
        local_dir=str(raw),
        # Skip test.tar (23 GB, no annotations).
        allow_patterns=[
            'splits_txt/*',
            'dataset/train.tar',
            'dataset/val.tar',
        ],
    )
    print('Download complete.')

In [ ]:
# Extract train.tar and val.tar into place. Skip macOS AppleDouble
# metadata files (`._foo`) that the original tar contains.

import tarfile
from tqdm.notebook import tqdm

def _is_appledouble(name: str) -> bool:
    return any(p.startswith('._') for p in name.split('/'))

for split in ('train', 'val'):
    tar_path = raw / 'dataset' / f'{split}.tar'
    target   = raw / 'dataset' / split
    if target.is_dir():
        print(f'  {split}: already extracted, skipping.')
        continue
    if not tar_path.exists():
        print(f'  {split}: {tar_path} not found, skipping.')
        continue
    print(f'  Extracting {tar_path.name}...')
    with tarfile.open(tar_path) as tf:
        members = [m for m in tf.getmembers() if not _is_appledouble(m.name)]
        for m in tqdm(members, desc=split):
            tf.extract(m, path=raw / 'dataset')
    print(f'  {split}: done.')

# Free up disk by removing the tars after successful extraction.
for split in ('train', 'val'):
    tar_path = raw / 'dataset' / f'{split}.tar'
    target   = raw / 'dataset' / split
    if target.is_dir() and tar_path.exists():
        tar_path.unlink()
        print(f'  removed {tar_path.name} ({split}/ is on disk)')

In [ ]:
# Sanity-check the SportsMOT layout we expect:
#   <SPORTSMOT_RAW>/splits_txt/{basketball,football,volleyball,train,val,test}.txt
#   <SPORTSMOT_RAW>/dataset/{train,val}/<seq>/{img1/, gt/gt.txt, seqinfo.ini}

print('Using SPORTSMOT_RAW =', SPORTSMOT_RAW)
print('Splits available:', sorted(p.name for p in (raw / 'splits_txt').glob('*.txt')))

# Cross-check basketball seq counts against split metadata.
bball = {l.strip() for l in (raw / 'splits_txt' / 'basketball.txt').read_text().splitlines() if l.strip()}
for split in ('train', 'val'):
    d = raw / 'dataset' / split
    if not d.exists():
        print(f'  {split}: NOT FOUND (extraction may have failed)')
        continue
    seqs = [p.name for p in d.iterdir() if p.is_dir()]
    n_bball = sum(1 for s in seqs if s in bball)
    print(f'  {split}: {len(seqs)} total seqs, {n_bball} basketball')

## 3. Convert basketball subset to YOLO format

This is the only step that touches per-frame data. It writes ~tens of
thousands of small label files plus symlinks back to the original images,
so it should finish in a couple of minutes on a T4-class CPU.

In [ ]:
from src.data.mot_to_yolo import convert

stats = convert(
    sportsmot_root=Path(SPORTSMOT_RAW),
    out_root=Path(SPORTSMOT_YOLO),
    sport='basketball',
    copy=False,   # symlinks are far faster; switch to True if your FS rejects them
)
for k, v in stats.items():
    print(f'{k:>16}: {v:,}')

## 4. Sanity-check: list a few outputs and render an annotated frame

In [ ]:
import random
yolo = Path(SPORTSMOT_YOLO)
for split in ('train', 'val'):
    imgs = sorted((yolo / 'images' / split).glob('*.jpg'))
    lbls = sorted((yolo / 'labels' / split).glob('*.txt'))
    print(f'{split}: {len(imgs):>6} images, {len(lbls):>6} labels')

sample_label = random.choice(list((yolo / 'labels' / 'train').glob('*.txt')))
print('\nExample label file:', sample_label.name)
print(sample_label.read_text()[:400])

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.viz.overlay import draw_on_frame

# Pick a frame that actually has labels (i.e. its .txt is non-empty).
lbl_paths = [p for p in (yolo / 'labels' / 'train').glob('*.txt') if p.stat().st_size > 0]
lbl = random.choice(lbl_paths)
img = yolo / 'images' / 'train' / (lbl.stem + '.jpg')
frame = cv2.imread(str(img))
h, w = frame.shape[:2]

# YOLO labels are normalized cx,cy,w,h. We need pixel x,y,w,h to draw.
boxes = []
for i, line in enumerate(lbl.read_text().strip().splitlines()):
    _, cx, cy, bw, bh = (float(t) for t in line.split())
    px = (cx - bw / 2) * w
    py = (cy - bh / 2) * h
    boxes.append((i + 1, px, py, bw * w, bh * h))   # use the row index as a fake track id

annotated = draw_on_frame(frame.copy(), boxes)
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f'{img.name} — {len(boxes)} player(s)')
plt.axis('off')
plt.show()

## 5. Done

If the stats look reasonable (thousands of frames in train, hundreds in
val, sane label files, an annotated frame that actually has tight boxes
around the players) then you're ready to switch the runtime to **A100**
and run `03_train_yolo.ipynb`.

**Common things to check before moving on:**
- The number of basketball sequences should be ~80 (varies slightly by
  release version of SportsMOT).
- The annotated frame should show tight boxes around players only — not
  referees, not coaches, not the ball.
- Empty label files for unlabeled frames are expected and intentional;
  YOLO uses them as true negatives during training.